<a href="https://colab.research.google.com/github/juliocnsouzadev/notebooks/blob/issue-17-Deep_Learning_for_Text_with_PyTorch/PyTorch/Deep_Learning_for_Text_With_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Text with PyTorch

In [ ]:
!pip uninstall torch -y
!pip install torch==2.0.0+cpu

!pip uninstall torchtext -y
!pip install torchtext==0.15.1

!pip uninstall nltk -y
!pip install nltk==3.6.5

!pip uninstall spacy -y
!pip install spacy==3.1.3

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

## Text Classification

In [1]:
import torch.nn as nn
import torch.nn.functional as F

class TextClassificationCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(TextClassificationCNN, self).__init__()
        # Initialize the embedding layer 
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv = nn.Conv1d(embed_dim, embed_dim, kernel_size=3, stride=1, padding=1)
        self.fc = nn.Linear(embed_dim, 2)
    def forward(self, text):
        embedded = self.embedding(text).permute(0, 2, 1)
        # Pass the embedded text through the convolutional layer and apply a ReLU
        conved = F.relu(self.conv(embedded))
        conved = conved.mean(dim=2) 
        return self.fc(conved)

In [ ]:
import torch
import torch.optim as optim

data = [(['I', 'love', 'this', 'book'], 1), (['This', 'is', 'an', 'amazing', 'novel'], 1), (['I', 'really', 'like', 'this', 'story'], 1), (['I', 'do', 'not', 'like', 'this', 'book'], 0), (['I', 'hate', 'this', 'novel'], 0), (['This', 'is', 'a', 'terrible', 'story'], 0)]

word_to_ix = {'I': 0, 'love': 1, 'this': 2, 'book': 3, 'This': 4, 'is': 5, 'an': 6, 'amazing': 7, 'novel': 8, 'really': 9, 'like': 10, 'story': 11, 'do': 12, 'not': 13, 'hate': 14, 'a': 15, 'terrible': 16}

model = TextClassificationCNN(vocab_size=len(word_to_ix), embed_dim=10)

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(10):
    for sentence, label in data:
        # Clear the gradients
        model.zero_grad()
        sentence = torch.LongTensor([word_to_ix.get(w, 0) for w in sentence]).unsqueeze(0)
        label = torch.LongTensor([int(label)])
        outputs = model(sentence)
        loss = criterion(outputs, label)
        loss.backward()
        # Update the parameters
        optimizer.step()
print('Training complete!')

In [ ]:
book_reviews = [
    "I love this book".split(),
    "I do not like this book".split()
]
for review in book_reviews:
    # Convert the review words into tensor form
    input_tensor = torch.tensor([word_to_ix[w] for w in review], dtype=torch.long).unsqueeze(0)
    # Get the model's output
    outputs = model(input_tensor)
    # Find the index of the most likely sentiment category
    _, predicted_label = torch.max(outputs.data, 1)
    # Convert the predicted label into a sentiment string
    sentiment = "Positive" if predicted_label.item() == 1 else "Negative"
    print(f"Book Review: {' '.join(review)}")
    print(f"Sentiment: {sentiment}\n")